In [1]:
!pip install py-vncorenlp transformers sentencepiece 

In [2]:
import os
import shutil
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel
from tqdm.auto import tqdm
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import autocast, GradScaler
import py_vncorenlp

# --- IMPORT LOẠI LORA ---
from peft import get_peft_model, LoraConfig, TaskType

class Config:
    TRAIN_PATH = r'/kaggle/input/neu-esc-dataset/train_set.csv'
    VAL_PATH = r'/kaggle/input/neu-esc-dataset/val_set.csv'
    TEST_PATH = r'/kaggle/input/neu-esc-dataset/test_set.csv' 
    
    MODEL_NAME = "VietAI/vit5-base"
    
    MAX_LEN = 512
    BATCH_SIZE = 4          
    ACCUMULATION_STEPS = 2  
    EPOCHS = 20
    LEARNING_RATE = 3e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    RDR_PATH = r'/kaggle/working/'

2026-01-15 02:41:23.145294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768444883.166869     184 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768444883.173424     184 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768444883.190449     184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768444883.190471     184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768444883.190474     184 computation_placer.cc:177] computation placer alr

In [3]:
class MultiTaskViT5_LoRA(nn.Module):
    def __init__(self, n_sentiment, n_topic):
        super(MultiTaskViT5_LoRA, self).__init__()
        print(f"Loading {Config.MODEL_NAME} with Optimized LoRA...")
        
        self.t5_encoder = T5EncoderModel.from_pretrained(Config.MODEL_NAME)
        
        peft_config = LoraConfig(
            r=16,                      
            lora_alpha=32,             
            target_modules=["q", "k", "v", "o", "wi", "wo"], 
            lora_dropout=0.1,           
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION 
        )
        
        self.t5_encoder = get_peft_model(self.t5_encoder, peft_config)
        self.t5_encoder.print_trainable_parameters()
        
        hidden_size = self.t5_encoder.config.d_model
        self.drop = nn.Dropout(p=0.15) # Tăng dropout ở đầu ra
        self.sent_out = nn.Linear(hidden_size, n_sentiment)
        self.topic_out = nn.Linear(hidden_size, n_topic)

    def forward(self, input_ids, attention_mask):
        outputs = self.t5_encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        output = self.drop(mean_pooled)
        return self.sent_out(output), self.topic_out(output)

def train_epoch(model, dataloader, optimizer, device, scaler, w_sent, w_topic):
    model.train()
    total_loss = 0
    
    crit_sent  = nn.CrossEntropyLoss(weight=w_sent, label_smoothing=0.1)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic, label_smoothing=0.1)
    
    optimizer.zero_grad()
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        sent_lbl = batch['sentiment_label'].to(device)
        topic_lbl = batch['topic_label'].to(device)

        with torch.amp.autocast('cuda'): 
            sent_logits, topic_logits = model(input_ids, mask)
            
            loss_s = crit_sent(sent_logits, sent_lbl)
            loss_t = crit_topic(topic_logits, topic_lbl)
            
            loss = (loss_s + 1.0 * loss_t) / Config.ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        
        if (step + 1) % Config.ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        current_loss = loss.item() * Config.ACCUMULATION_STEPS
        total_loss += current_loss
        
        del input_ids, mask, sent_lbl, topic_lbl, sent_logits, topic_logits, loss
        progress_bar.set_postfix({'loss': f"{current_loss:.4f}"})
        
    return total_loss / len(dataloader)

In [4]:
def get_class_weights(y, device):
    class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.tensor(class_weights, dtype=torch.float).to(device)


In [5]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights(y, device):
    classes = np.unique(y)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return torch.tensor(class_weights, dtype=torch.float).to(device)

def validate_epoch(model, dataloader, device, w_sent, w_topic):
    model.eval()
    total_loss = 0
    correct_sent = 0
    correct_topic = 0
    total_samples = 0
    
    crit_sent  = nn.CrossEntropyLoss(weight=w_sent, label_smoothing=0.05)
    crit_topic = nn.CrossEntropyLoss(weight=w_topic, label_smoothing=0.1)
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            sent_lbl = batch['sentiment_label'].to(device)
            topic_lbl = batch['topic_label'].to(device)

            sent_logits, topic_logits = model(input_ids, mask)
            
            loss = crit_sent(sent_logits, sent_lbl) + crit_topic(topic_logits, topic_lbl)
            total_loss += loss.item()

            pred_sent = torch.argmax(sent_logits, dim=1)
            correct_sent += (pred_sent == sent_lbl).sum().item()
            
            pred_topic = torch.argmax(topic_logits, dim=1)
            correct_topic += (pred_topic == topic_lbl).sum().item()
            
            total_samples += sent_lbl.size(0)
            
    return total_loss / len(dataloader), correct_sent / total_samples, correct_topic / total_samples

In [6]:
import py_vncorenlp
import os

if not os.path.exists(Config.RDR_PATH):
    os.makedirs(Config.RDR_PATH, exist_ok=True)
    try:
        py_vncorenlp.download_model(save_dir=Config.RDR_PATH)
    except:
        pass 

try:
    rdrsegmenter = py_vncorenlp.VnCoreNLP(save_dir=Config.RDR_PATH, annotators=["wseg"])
    print("✅ RDRSegmenter đã sẵn sàng.")
except Exception as e:
    print(f"❌ Lỗi khởi tạo VnCoreNLP: {e}")

def segment_text(text):
    if not isinstance(text, str) or not text.strip(): return ""
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text

❌ Lỗi khởi tạo VnCoreNLP: Please download the VnCoreNLP model!


In [7]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = df['text'].values
        self.sentiments = df['sentiment'].values
        self.topics = df['classification'].values 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.texts[index])
        # Gọi hàm segment_text đã định nghĩa ở bước trước
        text = segment_text(raw_text) 
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'sentiment_label': torch.tensor(self.sentiments[index], dtype=torch.long),
            'topic_label': torch.tensor(self.topics[index], dtype=torch.long)
        }

In [8]:
if __name__ == '__main__':
    # 1. Load Data (Copy lại logic load data của bạn)
    print("--- 1. Loading Data ---")
    try:
        df_train = pd.read_csv(Config.TRAIN_PATH)
        df_val = pd.read_csv(Config.VAL_PATH)
    except:
        # Dummy cho debug
        df_train = pd.DataFrame({'text': ['Test'], 'sentiment': [0], 'classification': [0]})
        df_val = df_train.copy()

    n_sent = int(df_train['sentiment'].max()) + 1
    n_topic = int(df_train['classification'].max()) + 1
    
    # 2. Tokenizer & Dataset
    print("\n--- 2. Tokenizer & Dataset ---")
    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    
    train_ds = MultiTaskDataset(df_train, tokenizer, Config.MAX_LEN)
    val_ds = MultiTaskDataset(df_val, tokenizer, Config.MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, num_workers=0)
    
    # 3. Class Weights
    w_sent = get_class_weights(df_train['sentiment'].values, Config.DEVICE)
    w_topic = get_class_weights(df_train['classification'].values, Config.DEVICE)

    # 4. Khởi tạo Model LoRA
    print("\n--- 3. Initializing ViT5 LoRA ---")
    model = MultiTaskViT5_LoRA(n_sent, n_topic) # Dùng class mới
    model.to(Config.DEVICE)
    
    # Lưu ý: Khi dùng LoRA, chỉ pass các tham số require_grad=True vào optimizer
    # Mặc dù AdamW tự lọc, nhưng làm rõ ràng sẽ tốt hơn
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=Config.LEARNING_RATE)
    
    scaler = GradScaler() 

    print("\n--- 4. Start Training (LoRA) ---")
    best_val_loss = float('inf')
    
    for epoch in range(Config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, Config.DEVICE, scaler, w_sent, w_topic)
        
        # Valid
        val_loss, val_acc_s, val_acc_t = validate_epoch(model, val_loader, Config.DEVICE, w_sent, w_topic)
        
        print(f" -> Train Loss: {train_loss:.4f}")
        print(f" -> Val Loss  : {val_loss:.4f} | Sent Acc: {val_acc_s:.4f} | Topic Acc: {val_acc_t:.4f}")
        
        # Save Best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Khi lưu LoRA model, tốt nhất nên lưu state_dict như bình thường với custom module này
            torch.save(model.state_dict(), "vit5_lora_best.pth")
            print(" ✅ Saved Best LoRA Model")

--- 1. Loading Data ---

--- 2. Tokenizer & Dataset ---

--- 3. Initializing ViT5 LoRA ---
Loading VietAI/vit5-base with Optimized LoRA...
trainable params: 2,654,208 || all params: 115,330,176 || trainable%: 2.3014

--- 4. Start Training (LoRA) ---

Epoch 1/20


/tmp/ipykernel_184/873345798.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Training:   0%|          | 0/5762 [00:00<?, ?it/s]

Validating:   0%|          | 0/827 [00:00<?, ?it/s]

 -> Train Loss: 4.6439
 -> Val Loss  : 4.1149 | Sent Acc: 0.7428 | Topic Acc: 0.7089
 ✅ Saved Best LoRA Model

Epoch 2/20


...

Validating:   0%|          | 0/827 [00:00<?, ?it/s]

 -> Train Loss: 4.0201
 -> Val Loss  : 4.0637 | Sent Acc: 0.7909 | Topic Acc: 0.7480

Epoch 20/20


Training:   0%|          | 0/5762 [00:00<?, ?it/s]

Validating:   0%|          | 0/827 [00:00<?, ?it/s]

 -> Train Loss: 4.0372
 -> Val Loss  : 4.1296 | Sent Acc: 0.7374 | Topic Acc: 0.7228


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import torch
from tqdm.auto import tqdm

ID2SENTIMENT = {
    0: "Neutral", 1: "Positive", 2: "Negative", 3: "Toxic"
}
ID2TOPIC = {
    0: "Spam", 1: "News", 2: "Academic", 3: "Other", 4: "Service",
    5: "Jobs & Recruitment", 6: "Personal Affairs", 7: "Social Affairs",
    8: "Help & Share", 9: "Club & Events"
}


print("--- 1. Loading & Processing Test Data ---")


TEST_PATH = "/kaggle/input/neu-esc-dataset/test_set.csv"
try:
    df_test = pd.read_csv(TEST_PATH)
    print(f"Test size raw: {len(df_test)}")
    
    print("⏳ Đang tách từ cho tập Test...")
    tqdm.pandas()
    df_test['text'] = df_test['text'].progress_apply(lambda x: segment_text(str(x)))
    
except Exception as e:
    print(f"❌ Lỗi load file test: {e}")
    df_test = pd.DataFrame({
        'text': ['Trường này học phí đắt quá', 'Thầy cô rất nhiệt tình'],
        'sentiment': [2, 1],
        'classification': [4, 2]
    })
    df_test['text'] = df_test['text'].apply(lambda x: segment_text(x))

# Tạo Dataset & Loader
test_ds = MultiTaskDataset(df_test, tokenizer, Config.MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)
# Load model để test
print("\n--- Loading Best LoRA Model for Testing ---")
model_test = MultiTaskViT5_LoRA(n_sent, n_topic)
model_test.to(Config.DEVICE)

# Load weights
try:
    model_test.load_state_dict(torch.load("vit5_lora_best.pth"))
    print("✅ Đã load weight LoRA thành công!")
except Exception as e:
    print(f"❌ Lỗi load weight: {e}")

def evaluate_test(model, dataloader, device):
    model.eval()
    true_sents, pred_sents = [], []
    true_topics, pred_topics = [], []
    
    print("Đang chạy đánh giá trên tập Test...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            labels_s = batch['sentiment_label'].cpu().numpy()
            labels_t = batch['topic_label'].cpu().numpy()
            
            # Forward
            logits_s, logits_t = model(input_ids, mask)
            
            # Predict
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            
            true_sents.extend(labels_s)
            pred_sents.extend(preds_s)
            true_topics.extend(labels_t)
            pred_topics.extend(preds_t)
            
    return true_sents, pred_sents, true_topics, pred_topics

y_true_s, y_pred_s, y_true_t, y_pred_t = evaluate_test(model_test, test_loader, Config.DEVICE)

# --- SENTIMENT REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")

# Lấy danh sách ID và Name đầy đủ
s_ids = sorted([i for i in ID2SENTIMENT.keys() if i < n_sent])
s_names = [ID2SENTIMENT[i] for i in s_ids]

# THÊM tham số labels=s_ids để cố định danh sách lớp
print(classification_report(y_true_s, y_pred_s, labels=s_ids, target_names=s_names, zero_division=0))

# --- TOPIC REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")

# Lấy danh sách ID và Name đầy đủ cho Topic
t_ids = sorted([i for i in ID2TOPIC.keys() if i < n_topic])
t_names = [ID2TOPIC[i] for i in t_ids]

# THÊM tham số labels=t_ids
print(classification_report(y_true_t, y_pred_t, labels=t_ids, target_names=t_names, zero_division=0))


def predict_single(text, model, tokenizer, device):
    model.eval()
    # 1. Tách từ
    text_seg = segment_text(text)
    # 2. Tokenize
    encoding = tokenizer.encode_plus(
        text_seg,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        l_s, l_t = model(input_ids, mask)
        p_s = torch.argmax(l_s, dim=1).item()
        p_t = torch.argmax(l_t, dim=1).item()
        
    return ID2SENTIMENT.get(p_s, "Unknown"), ID2TOPIC.get(p_t, "Unknown")

print("\n--- DEMO PREDICTION ---")
sample_txt = "Trường mình cơ sở vật chất tốt nhưng học phí hơi cao so với mặt bằng chung."
print(f"Input: {sample_txt}")

# Sửa model_best thành model_test (vì biến model_best không tồn tại trong scope này nếu chạy lại)
s, t = predict_single(sample_txt, model_test, tokenizer, Config.DEVICE) 
print(f"Sentiment: {s}")
print(f"Topic: {t}")

--- 1. Loading & Processing Test Data ---
Test size raw: 6613
⏳ Đang tách từ cho tập Test...


  0%|          | 0/6613 [00:00<?, ?it/s]


--- Loading Best LoRA Model for Testing ---
Loading VietAI/vit5-base with Optimized LoRA...
trainable params: 2,654,208 || all params: 115,330,176 || trainable%: 2.3014
✅ Đã load weight LoRA thành công!
Đang chạy đánh giá trên tập Test...


  0%|          | 0/1654 [00:00<?, ?it/s]


KẾT QUẢ: CẢM XÚC (SENTIMENT)
Accuracy: 0.7729

              precision    recall  f1-score   support

     Neutral       0.91      0.79      0.84      4558
    Positive       0.54      0.75      0.63       834
    Negative       0.59      0.72      0.65      1046
       Toxic       0.65      0.85      0.74       175

    accuracy                           0.77      6613
   macro avg       0.67      0.78      0.71      6613
weighted avg       0.80      0.77      0.78      6613


KẾT QUẢ: CHỦ ĐỀ (TOPIC)
Accuracy: 0.7422

                    precision    recall  f1-score   support

              Spam       0.38      0.61      0.47        84
              News       0.44      0.55      0.49       183
          Academic       0.82      0.81      0.81      2104
             Other       0.89      0.77      0.83      2882
           Service       0.52      0.71      0.60       474
Jobs & Recruitment       0.63      0.66      0.64       163
  Personal Affairs       0.40      0.61      0.49    

In [14]:
torch.save(model.state_dict(), "vit5_lora_last.pth")

In [ ]:
model_test = MultiTaskViT5_LoRA(n_sent, n_topic)
model_test.to(Config.DEVICE)

# Load weights
try:
    model_test.load_state_dict(torch.load("vit5_lora_last.pth"))
    print("✅ Đã load weight LoRA thành công!")
except Exception as e:
    print(f"❌ Lỗi load weight: {e}")

def evaluate_test(model, dataloader, device):
    model.eval()
    true_sents, pred_sents = [], []
    true_topics, pred_topics = [], []
    
    print("Đang chạy đánh giá trên tập Test...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            labels_s = batch['sentiment_label'].cpu().numpy()
            labels_t = batch['topic_label'].cpu().numpy()
            
            # Forward
            logits_s, logits_t = model(input_ids, mask)
            
            # Predict
            preds_s = torch.argmax(logits_s, dim=1).cpu().numpy()
            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            
            true_sents.extend(labels_s)
            pred_sents.extend(preds_s)
            true_topics.extend(labels_t)
            pred_topics.extend(preds_t)
            
    return true_sents, pred_sents, true_topics, pred_topics


y_true_s, y_pred_s, y_true_t, y_pred_t = evaluate_test(model_test, test_loader, Config.DEVICE)

# --- SENTIMENT REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CẢM XÚC (SENTIMENT)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_s, y_pred_s):.4f}\n")
s_names = [ID2SENTIMENT[i] for i in sorted(ID2SENTIMENT.keys()) if i < n_sent]
print(classification_report(y_true_s, y_pred_s, target_names=s_names, zero_division=0))

# --- TOPIC REPORT ---
print("\n" + "="*40)
print("KẾT QUẢ: CHỦ ĐỀ (TOPIC)")
print("="*40)
print(f"Accuracy: {accuracy_score(y_true_t, y_pred_t):.4f}\n")
t_names = [ID2TOPIC[i] for i in sorted(ID2TOPIC.keys()) if i < n_topic]
print(classification_report(y_true_t, y_pred_t, target_names=t_names, zero_division=0))

def predict_single(text, model, tokenizer, device):
    model.eval()
    # 1. Tách từ
    text_seg = segment_text(text)
    # 2. Tokenize
    encoding = tokenizer.encode_plus(
        text_seg,
        max_length=Config.MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        l_s, l_t = model(input_ids, mask)
        p_s = torch.argmax(l_s, dim=1).item()
        p_t = torch.argmax(l_t, dim=1).item()
        
    return ID2SENTIMENT.get(p_s, "Unknown"), ID2TOPIC.get(p_t, "Unknown")

print("\n--- DEMO PREDICTION ---")
sample_txt = "Đm trường như cái lờ tao vậy."
print(f"Input: {sample_txt}")
s, t = predict_single(sample_txt, model_test, tokenizer, Config.DEVICE)
print(f"Sentiment: {s}")
print(f"Topic: {t}")

Loading VietAI/vit5-base with Optimized LoRA...
trainable params: 2,654,208 || all params: 115,330,176 || trainable%: 2.3014
✅ Đã load weight LoRA thành công!
Đang chạy đánh giá trên tập Test...


  0%|          | 0/1654 [00:00<?, ?it/s]


KẾT QUẢ: CẢM XÚC (SENTIMENT)
Accuracy: 0.7311

              precision    recall  f1-score   support

     Neutral       0.93      0.70      0.80      4558
    Positive       0.47      0.82      0.60       834
    Negative       0.55      0.77      0.64      1046
       Toxic       0.57      0.83      0.67       175

    accuracy                           0.73      6613
   macro avg       0.63      0.78      0.68      6613
weighted avg       0.80      0.73      0.75      6613


KẾT QUẢ: CHỦ ĐỀ (TOPIC)
Accuracy: 0.7319

                    precision    recall  f1-score   support

              Spam       0.32      0.69      0.44        84
              News       0.45      0.48      0.46       183
          Academic       0.80      0.83      0.81      2104
             Other       0.90      0.72      0.80      2882
           Service       0.58      0.66      0.62       474
Jobs & Recruitment       0.59      0.64      0.62       163
  Personal Affairs       0.40      0.63      0.49    